# Threat group similarity — Jaccard matcher + Gemini explanation

**Project:** `mitre-attack-threat-group-similarity`  
**Course:** VILNIUS TECH, Gilusis mokymas, ND1 parts 1.2.1 / 1.2.2

**Problem.** Given only a list of observed MITRE ATT&CK technique IDs, find the most similar historical incidents and explain the match.

**Method (no embeddings).** Jaccard similarity between the input technique *set* and each incident's technique *set* in `data/incidents_strict.json`. An inverted index (`technique → incidents`) skips incidents with zero overlap. Gemini is called **once per query**, with the input IDs plus the top matches (incident ID, score, shared techniques), and writes a short explanation. Descriptions are never sent. If `GEMINI_API_KEY` is missing, the call is skipped and a template explanation is used.

This notebook is the system: **User → Notebook → Jaccard matcher → Incident store → Gemini → Output**.


## 1. System overview

| Step | Component | Input → output |
| --- | --- | --- |
| 1 | User | list of technique IDs, e.g. `["T1059.001", "T1071.001", "T1105"]` |
| 2 | Notebook | validate / normalize IDs |
| 3 | Jaccard matcher | top-k incident IDs + Jaccard scores |
| 4 | Incident store | technique lists (+ group); **no descriptions** |
| 5 | Gemini (`google-generativeai`) | one short explanation, or a template if there is no key |
| 6 | Output | JSON, table, bar chart of match % |

Jaccard score: \(|A \cap B| / |A \cup B|\). Match percentage is that score × 100.

### Component diagram


```mermaid
flowchart LR
    User["User"]
    subgraph NotebookSystem["Notebook"]
        JM["Jaccard matcher"]
        Gemini["Gemini"]
        Out["Output"]
    end
    Store[("Incident store")]

    User -->|"MITRE technique IDs"| NotebookSystem
    JM -->|"inverted index / technique sets"| Store
    Store -->|"candidate incidents"| JM
    JM -->|"top-k IDs + Jaccard scores"| Out
    Store -->|"lookup: techniques (+ group)"| Out
    Out -->|"IDs, scores, shared techniques"| Gemini
    Gemini -->|"short explanation"| Out
    Out -->|"JSON + bar chart"| User
```


### Sequence diagram

```mermaid
sequenceDiagram
    actor User
    participant Notebook
    participant Jaccard as Jaccard matcher
    participant Store as Incident store
    participant Gemini
    participant Output

    User->>Notebook: technique IDs
    Notebook->>Jaccard: match(techniques, k)
    Jaccard->>Store: candidates via inverted index
    Store-->>Jaccard: technique sets
    Jaccard-->>Notebook: top-k IDs + Jaccard scores

    Notebook->>Store: lookup IDs
    Store-->>Notebook: techniques (+ group if needed)

    alt GEMINI_API_KEY present
        Notebook->>Gemini: input techniques + matches (IDs, scores, shared techniques)
        Gemini-->>Notebook: short explanation
    else no key
        Notebook->>Notebook: template explanation
    end

    Notebook->>Output: matches + explanation
    Output-->>User: bar chart, table, JSON
```

PlantUML sources for both diagrams are in the repo `docs/` folder.


## 2. Setup

Install `google-generativeai` (the Gemini Python SDK used here) plus `matplotlib` / `pandas` for the chart and table. Then read **`GEMINI_API_KEY` from Colab Secrets**. If the secret is missing, later cells skip Gemini instead of crashing.


In [ ]:
import subprocess
import sys


def install_packages(packages):
    try:
        ipy = get_ipython()  # type: ignore[name-defined]
    except NameError:
        ipy = None
    if ipy is not None:
        ipy.run_line_magic("pip", "install -qU " + " ".join(packages))
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])


install_packages(["google-generativeai", "matplotlib", "pandas"])
print("Packages ready.")


In Colab: 🔑 Secrets → add `GEMINI_API_KEY` → enable notebook access. Do not paste the key into a cell.


In [ ]:
import os

GEMINI_API_KEY = None
IN_COLAB = False

try:
    from google.colab import userdata

    IN_COLAB = True
    try:
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception as exc:
        print(f"Colab Secret GEMINI_API_KEY not available ({type(exc).__name__}).")
        GEMINI_API_KEY = None
except ImportError:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not GEMINI_API_KEY:
        print("Not in Colab and no GEMINI_API_KEY env var.")

if isinstance(GEMINI_API_KEY, str):
    GEMINI_API_KEY = GEMINI_API_KEY.strip() or None

HAS_GEMINI = bool(GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.5-flash"
print("HAS_GEMINI =", HAS_GEMINI, "| model =", GEMINI_MODEL if HAS_GEMINI else "(skipped)")

if not IN_COLAB and not os.environ.get("DISPLAY"):
    import matplotlib

    matplotlib.use("Agg")


## 3. Incident store

Load `data/incidents_strict.json` from this GitHub repo (raw URL). Each record has `incident_id`, `group`, `techniques`, `description`. The store keeps an inverted index `technique → incident indices` and **`get()` omits descriptions**.


In [ ]:
import json
import re
import urllib.request
from collections import defaultdict
from pathlib import Path

REPO_OWNER = "MadAcolyte"
REPO_NAME = "mitre-attack-threat-group-similarity"
# Prefer a local checkout; then this PR branch (pre-merge Colab); then main.
RAW_BASES = [
    f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/cursor/jaccard-gemini-redesign-40b7",
    f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main",
]


def load_repo_json(relative_path: str):
    local = Path(relative_path)
    if local.is_file():
        print(f"Loaded {relative_path} from local {local}")
        return json.loads(local.read_text(encoding="utf-8"))

    last_error = None
    for base in RAW_BASES:
        url = f"{base}/{relative_path}"
        request = urllib.request.Request(
            url, headers={"User-Agent": "threat-group-similarity-notebook"}
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
            print(f"Loaded {relative_path} from {url}")
            return payload
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not load {relative_path}") from last_error


class IncidentStore:
    """Incident lookup + inverted index. Descriptions are not exposed."""

    def __init__(self, incidents: list[dict]):
        self.incidents = incidents
        self.by_id = {inc["incident_id"]: inc for inc in incidents}
        self.inverted: dict[str, set[int]] = defaultdict(set)
        for idx, inc in enumerate(incidents):
            for tid in inc["techniques"]:
                self.inverted[tid].add(idx)

    def get(self, incident_id: str) -> dict:
        inc = self.by_id[incident_id]
        return {
            "incident_id": inc["incident_id"],
            "group": inc["group"],
            "techniques": list(inc["techniques"]),
        }

    def candidates(self, techniques: list[str]) -> set[int]:
        hits: set[int] = set()
        for tid in techniques:
            hits |= self.inverted.get(tid, set())
        return hits


incidents = load_repo_json("data/incidents_strict.json")
prompt_spec = load_repo_json("prompts/prompt.json")
examples = load_repo_json("data/examples.json")
store = IncidentStore(incidents)
print(f"Incidents: {len(store.incidents)}")
print(f"Inverted index keys (unique techniques): {len(store.inverted)}")
print(f"Prompt: {prompt_spec.get('name')} v{prompt_spec.get('version')}")
print(f"(X, y) examples: {len(examples)}")
print("Sample record (no description):", json.dumps(store.get(incidents[0]["incident_id"]), indent=2)[:400])


## 4. Validate the input technique list

The only user input is MITRE technique IDs. Accept `T####` or `T####.###`, uppercase them, drop invalid tokens, dedupe, and warn about IDs that never appear in the incident store.


In [ ]:
TECHNIQUE_RE = re.compile(r"^T\d{4}(\.\d{3})?$", re.IGNORECASE)
KNOWN_TECHNIQUES = set(store.inverted)


def process_query(raw_techniques) -> dict:
    raw_list = list(raw_techniques)
    techniques: list[str] = []
    invalid: list[str] = []
    unknown: list[str] = []
    seen: set[str] = set()
    for raw in raw_list:
        token = str(raw).strip().upper()
        if not TECHNIQUE_RE.fullmatch(token):
            invalid.append(str(raw))
            continue
        if token in seen:
            continue
        seen.add(token)
        techniques.append(token)
        if token not in KNOWN_TECHNIQUES:
            unknown.append(token)
    if invalid:
        print("WARNING: dropped invalid IDs:", invalid)
    if unknown:
        print("WARNING: IDs not in incidents_strict.json:", unknown)
    return {"techniques": techniques, "invalid": invalid, "unknown": unknown}


print(process_query([" t1059.001 ", "T1059.001", "T1071.001", "T105", "T9999.001"]))


## 5. Jaccard matcher

For each incident that shares **at least one** technique with the query (via the inverted index):

\[
\mathrm{Jaccard}(Q, I) = \frac{|Q \cap I|}{|Q \cup I|}
\]

Return the top-k hits sorted by Jaccard (desc), then number of shared techniques, then `incident_id`. Each hit includes the incident ID, score, shared techniques, and group (needed for the `(X, y)` labels). No descriptions.


In [ ]:
DEFAULT_K = 5


def jaccard(query: set[str], incident: set[str]) -> float:
    if not query or not incident:
        return 0.0
    return len(query & incident) / len(query | incident)


class JaccardMatcher:
    def __init__(self, store: IncidentStore):
        self.store = store

    def match(self, techniques: list[str], k: int = DEFAULT_K) -> list[dict]:
        query = set(techniques)
        if not query:
            return []
        scored = []
        for idx in self.store.candidates(techniques):
            inc = self.store.incidents[idx]
            inc_set = set(inc["techniques"])
            shared = sorted(query & inc_set)
            score = jaccard(query, inc_set)
            details = self.store.get(inc["incident_id"])
            scored.append(
                {
                    "incident_id": details["incident_id"],
                    "group": details["group"],
                    "jaccard": round(score, 4),
                    "match_percentage": round(100.0 * score, 1),
                    "shared_techniques": shared,
                    "techniques": details["techniques"],
                }
            )
        scored.sort(
            key=lambda m: (-m["jaccard"], -len(m["shared_techniques"]), m["incident_id"])
        )
        return scored[:k]


matcher = JaccardMatcher(store)
demo = process_query(examples[0]["X"]["techniques"])
demo_hits = matcher.match(demo["techniques"], k=5)
print("Query:", demo["techniques"])
for hit in demo_hits:
    print(
        f"  {hit['jaccard']:.4f}  {hit['match_percentage']:5.1f}%  "
        f"{hit['incident_id']:22s}  {hit['group']:16s}  shared={hit['shared_techniques']}"
    )


## 6. Gemini — one explanation call

Fill `prompts/prompt.json` (`system_prompt` + `user_prompt_template`). The payload is **small**: the input technique list and, for each top hit, `incident_id`, Jaccard score, and shared technique IDs. No descriptions, no full technique dumps.

The SDK is **`google.generativeai`**. If there is no key (or the call fails), we skip Gemini and use a template that only restates the Jaccard evidence.


In [ ]:
def format_matches_for_prompt(matches: list[dict]) -> str:
    lines = []
    for m in matches:
        shared = ", ".join(m["shared_techniques"]) or "(none)"
        lines.append(
            f"- {m['incident_id']}  jaccard={m['jaccard']:.4f}  "
            f"match_percentage={m['match_percentage']}  shared=[{shared}]"
        )
    return "\n".join(lines) if lines else "(no matches)"


def build_user_prompt(techniques: list[str], matches: list[dict]) -> str:
    return (
        prompt_spec["user_prompt_template"]
        .replace("{{techniques}}", json.dumps(techniques))
        .replace("{{matches}}", format_matches_for_prompt(matches))
    )


def template_explanation(techniques: list[str], matches: list[dict]) -> str:
    if not matches:
        return (
            "Template explanation (Gemini skipped). "
            f"No incident in the store shares a technique with {techniques}."
        )
    top = matches[0]
    parts = [
        "Template explanation (Gemini skipped; no GEMINI_API_KEY or the API call failed).",
        f"Input techniques: {', '.join(techniques)}.",
        (
            f"Top incident {top['incident_id']} ({top['group']}) has Jaccard "
            f"{top['jaccard']:.4f} ({top['match_percentage']}%) because it shares "
            f"{', '.join(top['shared_techniques']) or 'no IDs'}."
        ),
    ]
    if len(matches) > 1:
        others = ", ".join(
            f"{m['incident_id']} {m['match_percentage']}%" for m in matches[1:3]
        )
        parts.append(f"Next hits: {others}.")
    return " ".join(parts)


def explain_with_gemini(techniques: list[str], matches: list[dict]) -> tuple[str, str]:
    """One Gemini call. Returns (text, backend) with backend gemini|template."""
    if not HAS_GEMINI:
        print("SKIP Gemini: no GEMINI_API_KEY. Using template explanation.")
        return template_explanation(techniques, matches), "template"

    try:
        import google.generativeai as genai

        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(
            model_name=GEMINI_MODEL,
            system_instruction=prompt_spec["system_prompt"],
        )
        user_prompt = build_user_prompt(techniques, matches)
        print(f"Calling {GEMINI_MODEL} once ({len(user_prompt)} chars) …")
        response = model.generate_content(user_prompt)
        text = (getattr(response, "text", None) or "").strip()
        if not text:
            raise RuntimeError("empty Gemini response")
        return text, "gemini"
    except Exception as exc:
        print(f"SKIP Gemini ({type(exc).__name__}: {exc}). Using template explanation.")
        return template_explanation(techniques, matches), "template"


print("Prompt preview:\n", build_user_prompt(demo["techniques"], demo_hits[:3])[:700])
demo_text, demo_backend = explain_with_gemini(demo["techniques"], demo_hits)
print("\nbackend:", demo_backend)
print(demo_text)


## 7. Output: `analyze()` + bar chart

`analyze(techniques)` is the full path: validate → Jaccard top-k → one Gemini (or template) explanation. The chart shows **match percentage** for the top hits of a query.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj, **kwargs):
        print(obj)


def analyze(techniques, k: int = DEFAULT_K) -> dict:
    query = process_query(techniques)
    matches = matcher.match(query["techniques"], k=k)
    explanation, backend = explain_with_gemini(query["techniques"], matches)
    return {
        "techniques": query["techniques"],
        "invalid": query["invalid"],
        "unknown": query["unknown"],
        "matches": matches,
        "explanation": explanation,
        "backend": backend,
    }


def matches_table(result: dict) -> pd.DataFrame:
    rows = []
    for m in result["matches"]:
        rows.append(
            {
                "incident_id": m["incident_id"],
                "group": m["group"],
                "match_percentage": m["match_percentage"],
                "jaccard": m["jaccard"],
                "shared_techniques": ", ".join(m["shared_techniques"]),
            }
        )
    return pd.DataFrame(rows)


def plot_match_percentages(result: dict, title: str, save_path=None):
    if not result["matches"]:
        print("No matches to plot.")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [m["incident_id"] for m in result["matches"]]
    values = [m["match_percentage"] for m in result["matches"]]
    ax.bar(labels, values, color="#3b6d9a")
    ax.set_ylim(0, 100)
    ax.set_ylabel("match % (Jaccard × 100)")
    ax.set_title(title)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=120)
    if IN_COLAB:
        plt.show()
    else:
        plt.close(fig)


result0 = analyze(examples[0]["X"]["techniques"])
print("backend:", result0["backend"])
print("explanation:", result0["explanation"])
display(matches_table(result0))
plot_match_percentages(result0, "Example 1 — top-k match %")


## 8. Three end-to-end examples (`data/examples.json`)

Each row is an `(X, y)` pair. `X.techniques` is the input. `y` is the **expected** Jaccard top incident/group computed from `incidents_strict.json` (not a Gemini answer). We run `analyze(X)`, print the explanation, plot match %, and check that the top incident ID equals `y.top_incident_id`.

Finally one summary chart: **top match % per example**.


In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("/opt/cursor/artifacts")
if not ARTIFACT_DIR.is_dir():
    ARTIFACT_DIR = Path("/tmp")

e2e = []
for i, example in enumerate(examples, start=1):
    x = example["X"]["techniques"]
    y = example["y"]
    print("\n" + "=" * 72)
    print(f"Example {i}  X = {x}")
    print(f"expected y.top_incident_id = {y['top_incident_id']}  group={y['top_group']}  {y['match_percentage']}%")
    result = analyze(x, k=5)
    e2e.append(result)
    top = result["matches"][0] if result["matches"] else {}
    ok = top.get("incident_id") == y["top_incident_id"]
    print("got     ", top.get("incident_id"), top.get("group"), top.get("match_percentage"), "%")
    print("matches y.top_incident_id:", ok)
    print("backend:", result["backend"])
    print("explanation:", result["explanation"])
    display(matches_table(result))
    plot_match_percentages(
        result,
        title=f"Example {i}: match %",
        save_path=ARTIFACT_DIR / f"example_{i}_match_percent.png",
    )

# Summary: one bar per example (top-1 match %).
fig, ax = plt.subplots(figsize=(7, 4))
labels = [f"ex{i}\n{r['matches'][0]['incident_id']}" for i, r in enumerate(e2e, 1)]
values = [r["matches"][0]["match_percentage"] for r in e2e]
ax.bar(labels, values, color="#3b6d9a")
ax.set_ylim(0, 100)
ax.set_ylabel("top match %")
ax.set_title("Match percentage per example")
plt.tight_layout()
summary_path = ARTIFACT_DIR / "match_percent_per_example.png"
fig.savefig(summary_path, dpi=120)
if IN_COLAB:
    plt.show()
else:
    plt.close(fig)
print("saved", summary_path)
print("All examples OK:", all(
    r["matches"][0]["incident_id"] == ex["y"]["top_incident_id"]
    for r, ex in zip(e2e, examples)
))


### Notes

- Matching is Jaccard only. There is no embedding model and no TF-IDF.
- Gemini is optional documentation on top of the ranked list. The `(X, y)` labels do not depend on it.
- Dataset copyright: © 2026 The MITRE Corporation. See the repository README.
